# Setup

First, we mount Google Drive to access files stored in it and set up the working directory where our menu images are located.

In [ ]:
# Mount Google Drive to access files
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# --- Working directory (where your project lives) ---
directory = '/content/drive/MyDrive/GenAI/RAG with OpenAI API/OpenAI Project'
os.chdir(directory)

In [ ]:
import os, base64
import pandas as pd
from pydantic import BaseModel
from openai import OpenAI

In [ ]:
# Retrieve the OpenAI API key from Colab's user data
from google.colab import userdata
openai_api_key = userdata.get('genai_course')

In [ ]:
# Set up the OpenAI client and specify the model to use
MODEL = "gpt-5"
client = OpenAI(api_key=openai_api_key)

In [ ]:
# --- Choose the folder with images (absolute path OR a subfolder name under `directory`) ---
images_input = input("Folder with images: ").strip()
IMAGE_DIR = images_input if os.path.isabs(images_input) else os.path.join(directory, images_input)

Folder with images (absolute or relative to project directory): Regatta Folder


# Helpful functions

In [ ]:
# Single helper: file -> data URL
def to_data_url(path: str) -> str:
    with open(path, "rb") as f:
        b64 = base64.b64encode(f.read()).decode("utf-8")
    return f"data:image/jpeg;base64,{b64}"

In [ ]:
# Process images
rows = []
image_files = sorted(
    f for f in os.listdir(IMAGE_DIR)
    if f.lower().endswith(('.png', '.jpg', '.jpeg'))
)

In [ ]:
class MenuItem(BaseModel):
    CategoryTitleDefault: str = ""
    SubcategoryTitleDefault: str = ""
    ItemNameDefault: str = ""
    ItemDescriptionDefault: str = ""
    ItemPrice: str = ""  # keep as text

In [ ]:
COLUMNS = list(MenuItem.model_fields.keys())
COLUMNS

['CategoryTitleDefault',
 'SubcategoryTitleDefault',
 'ItemNameDefault',
 'ItemDescriptionDefault',
 'ItemPrice']

In [ ]:
class MenuPayload(BaseModel):
    items: list[MenuItem]

In [ ]:
MenuPayload

__main__.MenuPayload

# Defining the System Prompt

In [ ]:
# --- System prompt (5 columns only) ---
system_prompt = """
You convert menu images into a structured Excel-ready dataset with EXACTLY these 5 columns.

Rules:
- Each row is one item.
- Leave fields as "" if unknown or not shown.
- Append only; do not remove anything.
- Output ONLY JSON that matches the schema exactly.

Columns and limits:
- CategoryTitleDefault: text, max 30 chars
- SubcategoryTitleDefault: text, max 30 chars ("" if none)
- ItemNameDefault: text, max 40 chars
- ItemDescriptionDefault: text, max 300 chars ("" if none)
- ItemPrice: decimal number as text, no currency symbol (e.g., "2.50" or "2,50")
"""

In [ ]:
for image in image_files:
    try:
        data_url = to_data_url(os.path.join(IMAGE_DIR, image))
        resp = client.responses.parse(
            model=MODEL,
            instructions=system_prompt,
            input=[{
                "role": "user",
                "content": [
                    {"type": "input_text", "text": "Extract menu items from this image. Output must match the schema exactly."},
                    {"type": "input_image", "image_url": data_url}
                ]
            }],
            text_format=MenuPayload,
        )
        payload = resp.output_parsed  # MenuPayload
        rows.extend([it.model_dump() for it in payload.items])
        print(f"✓ {image}: {len(payload.items)} item(s)")
    except Exception as e:
        print(f"⚠️ {image}: {e}")

✓ Regatta_page_1.jpg: 13 item(s)
✓ Regatta_page_2.jpg: 5 item(s)
✓ Regatta_page_3.jpg: 15 item(s)
✓ Regatta_page_4.jpg: 1 item(s)
✓ Regatta_page_5.jpg: 8 item(s)
✓ Regatta_page_6.jpg: 22 item(s)
✓ Regatta_page_7.jpg: 13 item(s)


In [ ]:
# After building `rows`:
df = pd.DataFrame([it.model_dump() for it in payload.items] if 'payload' in locals() else rows)
# After the loop
df = pd.DataFrame(rows).reindex(columns=COLUMNS)   # use all accumulated rows

In [ ]:
excel_name = input("Excel file name (without extension): ").strip() or "menu_export"
excel_path = os.path.join(directory, f"{excel_name}.xlsx")
df.to_excel(excel_path, index=False)
print(f"Excel saved: {excel_path}")

Excel file name (without extension): regatta
Excel saved: /content/drive/MyDrive/GenAI/RAG with OpenAI API/OpenAI Project/regatta.xlsx


In [ ]:
df

,CategoryTitleDefault,SubcategoryTitleDefault,ItemNameDefault,ItemDescriptionDefault,ItemPrice
0,Entradas,,Azeitonas,Olives,1.60
1,Entradas,,Cesto de Pão,Bread Basket,1.80
2,Entradas,,Creme de Legumes,Cream of Vegetables,3.00
3,Para Partilhar,,Nachos & Guacamole,,7.50
4,Para Partilhar,,REGATTA Tuna,Atum Fresco Marinado / Marinated Tuna,12.90
...,...,...,...,...,...
72,DIGESTIVOS & LICORES,,Moscatel Favaios,,2.00
73,DIGESTIVOS & LICORES,,Licor Beirão,,3.00
74,DIGESTIVOS & LICORES,,Vinho do Porto,,3.00
75,DIGESTIVOS & LICORES,,Whisky,,5.00
